In [ ]:
import os
from collections import defaultdict, OrderedDict

import dill
import numpy as np
import torch
from tqdm import tqdm

## Load checkpoint

In [ ]:
PATH = "/home/jovyan/malkolesov/checkpoints/pt_ckpts_archive/llama_ckpts/3.1/llama-3.1-70b-pretrain-128k_tokenizer_remapped-ba4500-rank0.pt"
ckpt = torch.load(
    PATH,
    mmap=True,
    map_location="cpu",
    pickle_module=dill
)

## Utility functions

In [ ]:
def extend_matrix(tensor, heads, dim, head_dim=128):
    split = tensor.split(head_dim, dim=dim)
    split += split[-heads:]
    return torch.concatenate(split, dim=dim)


def extend_matrix_zeros(tensor, heads, dim, head_dim=128):
    split = tensor.split(head_dim, dim=dim)
    for i in range(heads):
        split += (torch.zeros_like(split[0]),)
    return torch.concatenate(split, dim=dim)


def expand_attn(
    tensor: torch.Tensor,
    tensor_name,
    num_heads,
    num_kv_heads,
    num_added_heads,
    num_added_kv_heads,
    head_dim: int = 128
):
    if any(num is None for num in (num_heads, num_kv_heads, num_added_heads, num_added_kv_heads)):
        raise ValueError("Need to provide every head num and added head num!")

    if num_heads / num_kv_heads != num_added_heads / num_added_kv_heads:
        raise RuntimeError(
            "Added heads to kv head proportion must be the same as original! Got "
            f"{num_heads / num_kv_heads} for original and {num_added_heads / num_added_kv_heads} for added."
        )

    if 'qkv' in tensor_name:
        qkv = tensor
        q, k, v = torch.split(qkv, [num_heads * head_dim, num_kv_heads * head_dim, num_kv_heads * head_dim])
        
        q = extend_matrix(q, num_added_heads, 0)
        k = extend_matrix(k, num_added_kv_heads, 0)
        v = extend_matrix(v, num_added_kv_heads, 0)
        qkv = torch.concatenate([q,k,v])
        return qkv
    elif 'o_proj' in tensor_name:
        o_proj = extend_matrix_zeros(tensor, num_added_heads, 1)
        return o_proj
    
    return None


def extend_mlp(tensor: torch.Tensor, tensor_name, n_heads: int, base_hid_size: int = 1024):
    assert n_heads > 0
    
    if 'gate_proj' in tensor_name or 'up_proj' in tensor_name:
        assert tensor.shape[0] % base_hid_size == 0
        return extend_matrix(tensor, n_heads, 0, base_hid_size)
    elif 'down_proj' in tensor_name:
        assert tensor.shape[1] % base_hid_size == 0
        return extend_matrix_zeros(tensor, n_heads, 1, base_hid_size)
    return None


def add_llamaprostyle_layers(model_state_dict, prefix, insert_layers_dup_ids):
    if len(insert_layers_dup_ids) == 0:
        print("Insert layers ids list is empty, returning original state dict.")
        return model_state_dict
    
    def get_layer_idx(layer_name):
        if prefix not in layer_name:
            return None
        return int(layer_name.replace(prefix, '').split('.')[0])

    def change_layer_idx(layer_name, prev_idx, new_idx):
        return layer_name.replace(f".{prev_idx}.", f".{new_idx}.")

    # Get leyers and their names for each decoder block
    max_layer = 0
    blocks = defaultdict(list)
    for layer_name in model_state_dict.keys():
        layer_idx = get_layer_idx(layer_name)
        if layer_idx is None:
            blocks[layer_name].append(layer_name)
            continue

        max_layer = max(max_layer, layer_idx)
        blocks[str(layer_idx)].append(layer_name)

    # Update the blocks order
    blocks_names = list(blocks.keys())
    new_block_order = []
    for block_name in blocks_names:
        new_block_order.append(block_name)
        if block_name.isdigit() and int(block_name) in insert_layers_dup_ids:
            new_block_order.append(f"{block_name}_dup")

    # Update and initialize new dict
    updated_state_dict = OrderedDict()
    layer_idx = 0
    for block_name in new_block_order:
        if block_name in blocks:
            for layer_name in blocks[block_name]:
                if block_name.isdigit():
                    updated_layer_name = change_layer_idx(
                        layer_name,
                        get_layer_idx(layer_name),
                        layer_idx
                    )
                    updated_state_dict[updated_layer_name] = model_state_dict[layer_name]
                else:
                    updated_state_dict[layer_name] = model_state_dict[layer_name]
        
        if "_dup" in block_name:
            block_name = block_name[:-4]
            for layer_name in blocks[block_name]:
                layer_weight = model_state_dict[layer_name]
                if 'down_proj' in layer_name or 'o_proj' in layer_name:
                    layer_weight = torch.zeros_like(layer_weight)

                updated_layer_name = change_layer_idx(
                    layer_name,
                    get_layer_idx(layer_name),
                    layer_idx
                )
                updated_state_dict[updated_layer_name] = layer_weight

        if block_name.isdigit():
            layer_idx += 1

    return updated_state_dict


def expand_model(
    model,
    expand_attn: bool = True,
    expand_mlp: bool = True,
    attn_expand_conf: dict = None,
    mlp_expand_conf: dict = None
):
    if attn_expand_conf is None:
        attn_expand_conf = dict()
    if mlp_expand_conf is None:
        mlp_expand_conf = dict()

    for layer_name in tqdm(model.keys()):
        ### expand attn ###
        res = None
        if expand_attn:
            res = expand_attn(model[layer_name], layer_name, **attn_expand_conf)
        if res is None and expand_mlp:
            res = extend_mlp(model[layer_name], layer_name, **mlp_expand_conf)

        if res is not None:
            model[layer_name] = res

    return model


def expand_optim(optim):
    for k in tqdm(optim.keys()):
        for sub_k in optim[k]:
            tensor = optim[k][sub_k]
            if len(tensor.shape) < 2:
                continue
            res = expand_attn(tensor, k) 
            if res is None:
                res = extend_mlp(tensor, k)
            if res is not None:
                optim[k][sub_k] = res
    return optim

## Expand model and add layers

In [ ]:
EXPAND_OPTIMIZER: bool = False
EXPAND_LAYERS: bool = True
EXPAND_ATTENTION: bool = False
EXPAND_MLP: bool = True
INSERT_DUPLICATE_LAYERES_IDS: list = [12, 13, 25, 26, 38, 39, 51, 52, 64, 65, 77, 78]

assert len(INSERT_DUPLICATE_LAYERES_IDS) == 12

model_config = ckpt["state"]["integrations"]["huggingface"]["model"]["config"]["content"]
NUM_HEADS = model_config["num_attention_heads"]
NUM_KV_HEADS = model_config["num_key_value_heads"]
INTERMEDIATE_SIZE = model_config["intermediate_size"]

NUM_ADDED_HEADS: int = None
NUM_ADDED_KV_HEADS: int = None
NEW_INTERMEDIATE_SIZE: int = 32*1024

mlp_extension_size = NEW_INTERMEDIATE_SIZE - INTERMEDIATE_SIZE
assert mlp_extension_size > 0, "MLP target intermediate size is less than current one!"
mlp_n_heads = mlp_extension_size // 1024
print("Num added 'mlp heads'", mlp_n_heads)

if EXPAND_LAYERS:
    attn_expand_conf = {
        "num_heads": NUM_HEADS,
        "num_kv_heads": NUM_KV_HEADS,
        "num_added_heads": NUM_ADDED_HEADS,
        "num_added_kv_heads": NUM_ADDED_KV_HEADS,
        "head_dim": 128
    }
    mlp_expand_conf = {
        "n_heads": mlp_n_heads,
        "base_hid_size": 1024
    }
    ckpt['state']['model'] = expand_model(
        ckpt['state']['model'],
        expand_attn=EXPAND_ATTENTION,
        expand_mlp=EXPAND_MLP,
        attn_expand_conf=attn_expand_conf,
        mlp_expand_conf=mlp_expand_conf
    )
    if EXPAND_OPTIMIZER:
        ckpt['state']['optimizers']['DecoupledAdamW']['state'] = expand_model(
            ckpt['state']['optimizers']['DecoupledAdamW']['state'],
            expand_attn=EXPAND_ATTENTION,
            expand_mlp=EXPAND_MLP,
            attn_expand_conf=attn_expand_conf,
            mlp_expand_conf=mlp_expand_conf
        )

ckpt['state']['model'] = add_llamaprostyle_layers(
    ckpt['state']['model'], 
    prefix='model.model.layers.', 
    insert_layers_dup_ids=INSERT_DUPLICATE_LAYERES_IDS
)
ckpt["state"]["integrations"]["huggingface"]["model"]["config"]["content"]["num_hidden_layers"] += len(INSERT_DUPLICATE_LAYERES_IDS)

In [ ]:
ckpt["state"]["model"]["model.model.layers.91.mlp.down_proj.weight"].shape

In [ ]:
if EXPAND_LAYERS:
    raise RuntimeError("Layers were expanded, need to update checkpoint config params! See the next cell for that.")

In [ ]:
config = ckpt["state"]["integrations"]["huggingface"]["model"]["config"]["content"]
# config["num_attention_heads"] = ...
# config["num_kv_heads"] = ...
config["intermediate_size"] = 32 * 1024
# config["attention_hidden_size"] = ...

In [ ]:
config

In [ ]:
config["tp_size"] = 1

---

In [ ]:
# ckpt["state"]["model"].keys()

## Save updated checkpoint

In [ ]:
SAVE_PATH: str = "/home/jovyan/malkolesov/checkpoints/pt_ckpts_archive/llama_ckpts/3.1/llama-3.1-70b-pretrain-128k_tokenizer_remapped-ba4500-extended_to_80b-rank0.pt"
SAVE_OVERRIDE: bool = False
SAVE_MODEL_ONLY: bool = True

In [ ]:
if SAVE_PATH is None or SAVE_PATH == "":
    raise RuntimeError("Save path is not set or empty!")

if SAVE_MODEL_ONLY:
    print("Removing all the keys from the checkpoints except model, integrations and metadata...")
    for key in list(ckpt["state"].keys()):
        if key not in ["model", "integrations", "metadata"]:
            del ckpt["state"][key]

if os.path.exists(SAVE_PATH) and not SAVE_OVERRIDE:
    raise RuntimeError("Saved checkpoint already exists, but SAVE_OVERRRIDE is set to False!")
else:
    torch.save(ckpt, SAVE_PATH)